# 2. ARVO Router 평가

학습된 `router-arvo.pkl`을 불러와 project-disjoint dev/test split에서 평가합니다. 이 노트북도 OpenRouter API를 호출하지 않습니다.

In [1]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'arvo'
MODEL_PATH = ROOT / 'models' / 'router-arvo.pkl'

from llm_security.config import AppConfig
from llm_security.datasets import load_router_samples_jsonl
from llm_security.models import to_dict
from llm_security.routing import AdaptiveExpertRouter

## 데이터 누수 확인과 모델 로드

In [2]:
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
project_sets = {
    split: set(details['projects'])
    for split, details in manifest['splits'].items()
}
assert project_sets['train'].isdisjoint(project_sets['dev'])
assert project_sets['train'].isdisjoint(project_sets['test'])
assert project_sets['dev'].isdisjoint(project_sets['test'])
if not MODEL_PATH.exists():
    raise FileNotFoundError('Run 01_train_router.ipynb first.')
router = AdaptiveExpertRouter.load(MODEL_PATH)
dev_samples = load_router_samples_jsonl(DATA_DIR / 'router_dev.jsonl')
test_samples = load_router_samples_jsonl(DATA_DIR / 'router_test.jsonl')
print('project leakage: none')
print('dev/test samples:', len(dev_samples), len(test_samples))

project leakage: none
dev/test samples: 382 472


## Dev/Test 지표

In [3]:
config = AppConfig.from_env(ROOT / '.env')
calibration = router.calibrate_policy(
    dev_samples, target_coverage=config.router.target_coverage
)
router.save(MODEL_PATH)
dev_metrics = router.evaluate(dev_samples)
test_metrics = router.evaluate(test_samples)
summary = {
    'model': str(MODEL_PATH),
    'policy_calibration': to_dict(calibration),
    'dev': to_dict(dev_metrics),
    'test': to_dict(test_metrics),
}
pprint(summary)
metrics_path = ROOT / 'models' / 'router-arvo-metrics.json'
metrics_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('saved metrics:', metrics_path)

{'dev': {'adaptive_coverage': 0.6073298429319371,
         'average_experts_per_candidate': 1.968586387434555,
         'average_top1_confidence': 0.39325333868147344,
         'average_top1_top2_margin': 0.08473126327752041,
         'coverage_at_1': 0.2486910994764398,
         'coverage_at_2': 0.6073298429319371,
         'expected_calibration_error': 0.15466127752250106,
         'llm_calls_saved': 1540,
         'routing_accuracy': 0.2486910994764398,
         'routing_latency_ms': 0.38833874344815994,
         'sample_count': 382},
 'model': 'C:\\Users\\junhyun111\\Desktop\\llm-security\\models\\router-arvo.pkl',
 'policy_calibration': {'achieved_coverage': 0.6073298429319371,
                        'average_experts_per_candidate': 1.968586387434555,
                        'high_confidence': 0.5,
                        'min_margin': 0.4,
                        'target_coverage': 0.95,
                        'target_met': False},
 'test': {'adaptive_coverage': 0.5847457627118

## Test 예측 일부 확인

In [4]:
pair_counts = Counter()
for sample in test_samples:
    decision = router.route(sample.candidate)
    expected = sample.labels[0].value
    selected = decision.selected[0].value
    pair_counts[(expected, selected)] += 1

print('expected -> selected counts:')
for pair, count in sorted(pair_counts.items()):
    print(pair, count)

print('\nfirst 20 predictions:')
for sample in test_samples[:20]:
    decision = router.route(sample.candidate)
    print(
        sample.candidate.project_id,
        sample.candidate.function,
        'expected=', [family.value for family in sample.labels],
        'selected=', [family.value for family in decision.selected],
    )

expected -> selected counts:
('control_state_error', 'control_state_error') 24
('control_state_error', 'lifetime_resource') 129
('control_state_error', 'memory_bounds') 47
('lifetime_resource', 'control_state_error') 9
('lifetime_resource', 'lifetime_resource') 29
('lifetime_resource', 'memory_bounds') 10
('memory_bounds', 'control_state_error') 29
('memory_bounds', 'lifetime_resource') 137
('memory_bounds', 'memory_bounds') 58

first 20 predictions:
c-blosc2 ndlz8_decompress expected= ['memory_bounds'] selected= ['control_state_error', 'memory_bounds']
c-blosc2 LLVMFuzzerTestOneInput expected= ['control_state_error'] selected= ['lifetime_resource', 'memory_bounds']
c-blosc2 zlib_wrap_decompress expected= ['memory_bounds'] selected= ['lifetime_resource', 'memory_bounds']
c-blosc2 zlib_wrap_decompress expected= ['memory_bounds'] selected= ['lifetime_resource', 'memory_bounds']
c-blosc2 blosc_d expected= ['control_state_error'] selected= ['control_state_error', 'lifetime_resource']
c-blo